# PC to z to CAD

In this notebook I want to develop the pipeline from point cloud to CAD-model. To do that I will use my trained PointNet++ to encode a point cloud into the latent represenation z and then the DeepCAD decoder will reconstruct the CAD model from this.

In [1]:
import sys
import importlib
import os
import torch
import open3d as o3d

sys.path.append(os.path.abspath("../code"))
import dataset
from dataset import PointCloudEmbeddingDataset
importlib.reload(dataset)

<module 'dataset' from '/Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/code/dataset.py'>

In [2]:
def inplace_relu(m):
    classname = m.__class__.__name__
    if classname.find('ReLU') != -1:
        m.inplace=True

In [3]:
train_dataset = PointCloudEmbeddingDataset("../data", 'train')

### Loading train dataset ###

Number of samples that should be in the train set: 161240
Files on disk: 160982 --> There are 258 missing point cloud files in the train set.

Checking latent representation:
All latent represenations are valid.

--- DONE ---



In [4]:
index = 0

point_cloud = train_dataset[index][0]
latent_rep = train_dataset[index][1]
train_dataset.get_path(index)

'../data/pc_cad/0067/00675619.ply'

In [5]:
def visualize_pc(pc_path):
    point_cloud = o3d.io.read_point_cloud(pc_path)
    print(point_cloud)
    o3d.visualization.draw_geometries([point_cloud])
#visualize_pc(train_dataset.get_path(index))

## Plan
- import trained PN++ DONE
- infer PC with PN++ DONE
- compare predicted z with target z DONE (using MSE)
- import pretrained decoder
- infer z with decoder
- compare predicted CAD-sequence with target CAD-sequence
- visualize CAD model

In [6]:
sys.path.append(os.path.join("..", 'models','Pointnet_Pointnet2_pytorch', 'models'))
model = importlib.import_module('pointnet2_cls_ssg')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

classifier = model.get_model(256, normal_channel=False)
criterion = model.get_loss_mse()
classifier.apply(inplace_relu)

model_path = 'best.pth'
model_dict = torch.load(model_path, map_location=torch.device(device), weights_only=True)
state_dict = model_dict['model_state_dict']
classifier.load_state_dict(state_dict)
classifier = classifier.to(device)
classifier.eval()

print("Loaded model")

Loaded model


In [7]:
def pc_to_z(model, pc, target):
    with torch.no_grad():
        pc = pc.unsqueeze(1) # CRITICAL: shape = (B,1,256), NOT (1,B,256) -> unsqueeze(1 not 0)
        target = target.unsqueeze(0)
        pc = pc.transpose(2,1)
        pred, _ = classifier(pc)
        mse = criterion(pred,target)
        print(f"MSE: {mse:.4f}")
        

In [8]:
pc_to_z(classifier, point_cloud, latent_rep)

MSE: 0.1229


In [16]:
sys.path.append(os.path.abspath(".."))
from models.DeepCAD.trainer.trainerAE import TrainerAE

In [ ]:
print(sys.path)

In [38]:
import h5py
hehe = '../models/trained_models/test_run/results/testing/z.h5'

In [41]:
with h5py.File(hehe, 'r') as hf:
    print(hf['latent_rep'][8])

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
